# DataType - Python

All 12 Python examples from [docs/datatype.md](https://platob.github.io/yggdryl/datatype/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import DataType

value = DataType("decimal(18, 4)")
assert value == DataType.decimal(18, 4)

assert str(value) == "decimal128(18,4)"
assert DataType(str(value)) == value
assert DataType.from_json(value.into_json()) == value

## Children

In [ ]:
from yggdryl import DataType, Field, fields

quote = DataType.from_fields([
    Field("symbol", "utf8", nullable=False),
    fields.list("levels", fields.float64("item")),
])

assert len(quote) == 2
assert [field.name for field in quote] == ["symbol", "levels"]
assert quote[0].name == "symbol"
assert quote[-1].name == "levels"
assert len(quote["levels"].data_type) == 1
assert "levels" in quote and "missing" not in quote

lookup = fields.map_of("lookup", "utf8", "int64", keys_sorted=True).data_type
assert len(lookup) == 1
assert lookup[0].name == "entries"

## Precision and resolution pick the width

In [ ]:
import pytest

from yggdryl import DataType

assert DataType.decimal(38, 4) == DataType("decimal128(38,4)")
assert DataType.decimal(39, 4) == DataType("decimal256(39,4)")
assert DataType.time("s") == DataType("time32(s)")
assert DataType.time("nano seconds") == DataType("time64(ns)")

with pytest.raises(ValueError, match="positive scale cannot exceed precision"):
    DataType.decimal(2, 3)
with pytest.raises(ValueError, match="temporal resolution"):
    DataType.time("year_month")

## Encodings that wrap a value

In [ ]:
import pytest

from yggdryl import fields

codes = fields.dictionary("codes", "int16", "utf8").data_type
runs = fields.run_end_encoded(
    "runs",
    fields.int16("run_ends", nullable=False),
    fields.utf8("values"),
).data_type

assert codes.kind == "dictionary"
assert runs.kind == "run_end_encoded"
assert not codes.is_nested and not runs.is_nested
assert str(codes) == "dictionary(int16,utf8)"

with pytest.raises(ValueError, match="integer key datatype"):
    fields.dictionary("bad", "utf8", "utf8")
with pytest.raises(ValueError, match="int16, int32, or int64"):
    fields.run_end_encoded(
        "bad", fields.uint32("run_ends", nullable=False), fields.utf8("values")
    )

## Unions and the dense-union sugar

In [ ]:
import pytest

from yggdryl import DataType, Field

variant = DataType.variant([
    Field("number", "int64", nullable=False),
    Field("text", "utf8"),
])

assert variant.id == "union"
assert str(variant).startswith("union(dense,")
assert [field.name for field in variant] == ["number", "text"]
assert DataType("variant(number:int64,text:string)").id == "union"

with pytest.raises(ValueError, match="duplicate field name"):
    DataType.variant([Field("same", "int64"), Field("same", "utf8")])

## Variant, geometry, and geography

In [ ]:
import pytest

from yggdryl import DataType, fields

variant = DataType.variant()
assert variant.id == "variant"
assert variant.kind == "variant"
assert str(variant) == "variant"
assert DataType("variant") == variant
assert DataType("variant(n:int64)").id == "union"
assert fields.variant("payload").data_type == variant

geometry = DataType.geometry()
assert str(geometry) == "geometry"
assert geometry == DataType.geometry("OGC:CRS84")
assert geometry.kind == "geospatial"
assert str(DataType.geometry("EPSG:3857")) == 'geometry("EPSG:3857")'

geography = DataType.geography()
assert str(geography) == "geography"
assert geography == DataType.geography("OGC:CRS84", "spherical")

vincenty = DataType.geography("OGC:CRS84", "vincenty")
assert str(vincenty) == 'geography("OGC:CRS84","vincenty")'
assert DataType(str(vincenty)) == vincenty
assert fields.geography("region", "OGC:CRS84", "vincenty").data_type == vincenty

with pytest.raises(ValueError, match="expected no edge algorithm"):
    DataType("geometry('OGC:CRS84', 'vincenty')")
with pytest.raises(ValueError, match="expected one of spherical"):
    DataType.geography("OGC:CRS84", "euclidean")

## Identity and family

In [ ]:
from yggdryl import DataType

stamp = DataType("timestamp(ns, Europe/Paris)")
assert stamp.id == "timestamp"
assert stamp.kind == "temporal"

assert DataType("timestamp(s)").id == stamp.id
assert DataType("timestamp(s)") != stamp

assert DataType.decimal(38, 4).id == "decimal128"
assert DataType.decimal(38, 4).kind == "decimal"

## Arrow projection

In [ ]:
import pyarrow as pa

from yggdryl import DataType

value = DataType("map<string,array<decimal(38,18)>>")
arrow = value.into_arrow()

assert DataType.from_arrow(arrow) == value
assert DataType(arrow) == value
assert value.into_arrow() == arrow

assert DataType(pa.int64()) == DataType("int64")
assert DataType("int64").into_arrow() == pa.int64()

## Default values

In [ ]:
from yggdryl import DataType, Field

value = DataType.from_fields([
    Field("id", "int32", nullable=False),
    Field("note", "utf8", nullable=True),
])
row = value.default_pyvalue()

assert (row.id, row.note) == (0, None)
assert DataType("utf8").default_pyvalue() == ""
assert DataType("int64").default_pyhint() is int
assert value.default_arrow_scalar().as_py() == {"id": 0, "note": None}

## Serializing a schema

In [ ]:
from yggdryl import DataType

data_type = DataType.decimal(9, 2)

assert DataType.from_dict(data_type.into_dict()) == data_type
assert DataType.from_json(data_type.into_json()) == data_type
assert DataType.from_yaml(data_type.into_yaml()) == data_type
assert DataType.from_toml(data_type.into_toml()) == data_type

assert data_type.into_dict()["type"] == "decimal128"

## A readable rendering

In [ ]:
from yggdryl import DataType, Field

rows = DataType.from_fields([Field("venue", "utf8")])

# `repr` is unchanged - the eval-round-trip form Python expects.
assert repr(rows).startswith("DataType.from_str(")
assert DataType.from_str(str(rows)) == rows

assert rows.pretty() == "struct[1]\n  venue: utf8, nullable"

## Compatibility rewriting

In [ ]:
import pytest

from yggdryl import DataType, Field

source = DataType.from_fields([
    Field("small", "uint8", nullable=False),
    Field("wide", "uint64", nullable=True),
])

spark = source.into_scheme_compat("spark")
assert str(spark["small"].data_type) == "int16"
assert str(spark["wide"].data_type) == "decimal128(20,0)"

assert source.into_scheme_compat("arrow") == source
assert DataType("uint32").into_scheme_compat("polars") == DataType("uint32")

with pytest.raises(ValueError, match="got ns"):
    DataType("timestamp(ns)").into_scheme_compat("spark")
with pytest.raises(ValueError, match="arrow, spark, polars, pandas"):
    DataType("int32").into_scheme_compat("duckdb")